# MODUL PRAKTIKUM BIG DATA
## Pertemuan 6 — Format Data Besar (Parquet & JSON) dan Pipeline ETL Sederhana

| | |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Pertemuan** | 6 |
| **Topik** | Format Parquet, Membaca Data JSON, Membangun Pipeline ETL (Extract-Transform-Load) |
| **Estimasi Waktu** | 3 x 50 menit |
| **Prasyarat** | Modul Pertemuan 1-5 selesai (Hadoop + HDFS aktif, Spark/PySpark 3.5.9 terpasang, memahami join & window function) |

---

Tidak ada instalasi baru pada pertemuan ini — seluruh tools sudah lengkap sejak Pertemuan 1-4. Pastikan Hadoop aktif (`start-dfs.sh`, `start-yarn.sh`) sebelum memulai.

> **Konsistensi versi:** Tetap **Spark/PySpark 3.5.9**, **Python 3.11**, **Hadoop 3.4.3** — tidak ada perubahan versi apa pun pada pertemuan ini.


---
## Recap Pertemuan Sebelumnya

- [ ] Hadoop aktif (`jps` menampilkan 5 proses)
- [ ] Memahami `join()`, window function, dan Spark SQL dari Pertemuan 5

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 6, mahasiswa mampu:
1. Menjelaskan mengapa format **Parquet** lebih efisien dibanding CSV untuk kebutuhan Big Data.
2. Membaca dan menulis data dalam format Parquet dan JSON menggunakan PySpark.
3. Menerapkan **partitioning** saat menulis data untuk mengoptimalkan kueri di masa depan.
4. Membangun **pipeline ETL** (Extract, Transform, Load) sederhana dari beberapa sumber data hingga tersimpan siap pakai di HDFS.

---

## 6.1 Mengapa Parquet, Bukan Sekadar CSV?

Sejauh ini kita selalu memakai CSV — mudah dibaca manusia, namun ternyata **kurang efisien** untuk kebutuhan Big Data. **Apache Parquet** adalah format penyimpanan **kolumnar** (*columnar storage*) yang menjadi standar de facto di ekosistem Big Data modern.

| Aspek | CSV (row-based) | Parquet (columnar) |
|---|---|---|
| **Struktur penyimpanan** | Data disimpan per-baris | Data disimpan per-**kolom** |
| **Ukuran berkas** | Lebih besar, teks polos | Terkompresi otomatis, jauh lebih kecil |
| **Kecepatan baca sebagian kolom** | Harus membaca seluruh baris meski hanya butuh 1 kolom | Bisa langsung membaca kolom yang dibutuhkan saja (*column pruning*) |
| **Skema (tipe data)** | Tidak tersimpan — harus ditebak ulang (`inferSchema`) setiap kali dibaca | **Tersimpan built-in** di dalam berkas, tidak perlu ditebak ulang |
| **Dapat dipecah untuk diproses paralel** | Kurang optimal | Sangat optimal — dirancang untuk sistem terdistribusi |

> **Intuisi sederhana:** Bayangkan tabel Excel dengan 20 kolom, namun kalian hanya butuh 2 kolom untuk analisis. Format CSV tetap harus "membaca" ke-20 kolom setiap baris karena datanya berurutan per-baris. Format Parquet bisa langsung meloncat hanya ke 2 kolom yang dibutuhkan, karena setiap kolom disimpan terpisah secara fisik.

Mari kita buktikan sendiri keunggulan ukuran berkasnya, bukan hanya percaya begitu saja.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/24 11:09:03 WARN Utils: Your hostname, vmi3356457 resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/09/24 11:09:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 11:09:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


### 6.1.1 Membandingkan Ukuran CSV vs Parquet Secara Langsung

Agar perbedaannya benar-benar terlihat, kita perlu dataset yang **cukup besar** (dataset kecil di pertemuan sebelumnya tidak akan menunjukkan keunggulan Parquet secara meyakinkan — bahkan bisa jadi Parquet terlihat *lebih besar* karena overhead metadata pada data yang sangat sedikit).

In [2]:
# Membuat dataset yang lebih besar (20.000 baris) khusus untuk latihan ini
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"Ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

Ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [5]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 0 bytes (0.0 KB)
Parquet 100.0% lebih kecil dibanding CSV


Perbedaannya cukup signifikan bukan? Ini murni berkat **kompresi otomatis** dan **penyimpanan kolumnar** Parquet — semakin besar & semakin berulang nilai suatu data (seperti kolom `kategori`, `kota` yang hanya berisi beberapa nilai berbeda), semakin besar pula penghematannya.

### 6.1.2 Skema Otomatis Tersimpan di Parquet

In [6]:
# Membaca kembali Parquet — perhatikan kita TIDAK perlu inferSchema=True
# karena skema/tipe data sudah tersimpan built-in di dalam berkas Parquet itu sendiri
df_dari_parquet = spark.read.parquet("data_transaksi_besar_parquet")
df_dari_parquet.printSchema()
df_dari_parquet.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



+--------+-----------+----------+---------+-----------------+----------+
|order_id|customer_id|  kategori|     kota|metode_pembayaran|pendapatan|
+--------+-----------+----------+---------+-----------------+----------+
|      O0|         17|   Makanan|     Solo|              COD|    196650|
|      O1|         18| Kesehatan| Magelang|         E-Wallet|    283512|
|      O2|         14|Elektronik| Semarang|              COD|    108835|
|      O3|         13| Kesehatan|Purworejo|     Kartu Kredit|    390604|
|      O4|          2|   Fashion|Purworejo|         E-Wallet|    427224|
+--------+-----------+----------+---------+-----------------+----------+
only showing top 5 rows



---
## 6.2 Partitioning saat Menulis Data

Saat menulis Parquet, kita dapat **mempartisi** data berdasarkan nilai kolom tertentu — Spark akan otomatis membuat sub-folder terpisah untuk setiap nilai unik kolom tersebut. Ini sangat mempercepat kueri di masa depan yang menyaring berdasarkan kolom itu, karena Spark bisa langsung meloncat ke folder yang relevan tanpa membaca seluruh dataset.

In [9]:
# Menulis ulang dengan partisi berdasarkan kolom 'kota'
df_besar.write.mode("overwrite").partitionBy("kota").parquet("data_transaksi_partisi_kota")

# Melihat struktur folder yang dihasilkan
!hdfs dfs -ls data_transaksi_partisi_kota

Found 6 items
-rw-r--r--   1 zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/_SUCCESS
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/kota=Magelang
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/kota=Purworejo
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/kota=Semarang
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/kota=Solo
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:24 data_transaksi_partisi_kota/kota=Yogyakarta


Perhatikan Spark otomatis membuat folder seperti `kota=Magelang`, `kota=Semarang`, dst. Ketika nanti kita membaca dan menyaring `WHERE kota = 'Magelang'`, Spark **hanya perlu membuka folder `kota=Magelang`** — tidak menyentuh folder kota lain sama sekali. Teknik ini disebut ***partition pruning***, dan menjadi salah satu alasan utama mengapa Parquet + partitioning adalah kombinasi standar di seluruh platform Big Data (termasuk Hadoop, Spark, maupun layanan cloud seperti yang kalian pelajari di mata kuliah Komputasi Awan).

---

## 6.3 Membaca Data JSON (Semi-Terstruktur)

Selain data tabular (CSV/Parquet), Big Data sering melibatkan data **semi-terstruktur** seperti JSON — misalnya data dari API, log aplikasi, atau data NoSQL. Mari berlatih membaca data pelanggan berformat JSON dan menggabungkannya dengan data transaksi.

In [10]:
import json

# Membuat data pelanggan sebagai JSON Lines (satu objek JSON per baris — format umum untuk big data)
np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"])
    }
    for i in range(1, 21)
]

with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

print("Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).")

Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).


In [12]:
# Membaca JSON menggunakan Spark — Spark otomatis mendeteksi struktur/skema JSON
df_pelanggan = spark.read.json("data_pelanggan.json")
df_pelanggan.printSchema()
df_pelanggan.show(5)

root
 |-- customer_id: long (nullable = true)
 |-- kota_domisili: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- nama: string (nullable = true)



+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silver|Pelanggan3|
|          4|         Solo|    Silver|Pelanggan4|
|          5|         Solo|      Gold|Pelanggan5|
+-----------+-------------+----------+----------+
only showing top 5 rows



> **Format JSON Lines vs JSON biasa:** Spark membaca JSON dengan asumsi **satu objek JSON per baris** (disebut *JSON Lines* atau *NDJSON*) — bukan satu array besar berisi banyak objek seperti file `.json` konvensional. Ini karena format per-baris jauh lebih mudah diproses secara paralel & terdistribusi (setiap baris independen), sesuai filosofi Big Data.


---
## 6.4 Membangun Pipeline ETL Sederhana

**ETL** (*Extract, Transform, Load*) adalah pola kerja paling fundamental dalam pengolahan Big Data:

| Tahap | Deskripsi | Pada praktikum ini |
|---|---|---|
| **Extract** | Mengambil data dari sumber (bisa banyak & berbeda format) | Membaca CSV transaksi + JSON pelanggan |
| **Transform** | Membersihkan, menggabungkan, memperkaya data | Join, menghitung kolom turunan, agregasi |
| **Load** | Menyimpan hasil ke tujuan akhir, siap dipakai | Menulis ke HDFS dalam format Parquet, terpartisi |

kita mulai pipeline ETL lengkap, menggabungkan seluruh skill dari Pertemuan 3-6.

### Tahap 1: EXTRACT

In [13]:
# EXTRACT — membaca dari berbagai sumber
df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")

print("Data transaksi:", df_transaksi_etl.count(), "baris")
print("Data pelanggan:", df_pelanggan_etl.count(), "baris")

Data transaksi: 20000 baris
Data pelanggan: 20 baris


### Tahap 2: TRANSFORM

In [14]:
from pyspark.sql.functions import when

# TRANSFORM — menggabungkan, membersihkan, dan memperkaya data
df_etl = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="left")

# Menambahkan kolom turunan: kategori nilai transaksi
df_etl = df_etl.withColumn(
    "segmen_transaksi",
    when(col("pendapatan") >= 300000, "Tinggi")
    .when(col("pendapatan") >= 150000, "Sedang")
    .otherwise("Rendah")
)

# Memastikan tidak ada data pelanggan yang gagal ter-join (data quality check sederhana)
jumlah_tanpa_pelanggan = df_etl.filter(col("nama").isNull()).count()
print(f"Transaksi tanpa data pelanggan yang cocok: {jumlah_tanpa_pelanggan}")

df_etl.select("order_id", "nama", "membership", "kategori", "pendapatan", "segmen_transaksi").show(5)

Transaksi tanpa data pelanggan yang cocok: 0
+--------+-----------+----------+----------+----------+----------------+
|order_id|       nama|membership|  kategori|pendapatan|segmen_transaksi|
+--------+-----------+----------+----------+----------+----------------+
|      O0|Pelanggan17|      Gold|   Makanan|    196650|          Sedang|
|      O1|Pelanggan18|  Platinum| Kesehatan|    283512|          Sedang|
|      O2|Pelanggan14|  Platinum|Elektronik|    108835|          Rendah|
|      O3|Pelanggan13|      Gold| Kesehatan|    390604|          Tinggi|
|      O4| Pelanggan2|    Silver|   Fashion|    427224|          Tinggi|
+--------+-----------+----------+----------+----------+----------------+
only showing top 5 rows



### Tahap 3: LOAD

In [16]:
# LOAD — menyimpan hasil akhir ke HDFS dalam format Parquet, terpartisi berdasarkan kategori
!hdfs dfs -mkdir -p /user/mahasiswa/etl_output

df_etl.write.mode("overwrite").partitionBy("kategori").parquet(
    "hdfs://localhost:9000/user/mahasiswa/etl_output/transaksi_enriched"
)

print("Pipeline ETL selesai — hasil tersimpan di HDFS.")
!hdfs dfs -ls /user/mahasiswa/etl_output/transaksi_enriched

Pipeline ETL selesai — hasil tersimpan di HDFS.
Found 6 items
-rw-r--r--   1 zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/_SUCCESS
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Elektronik
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Fashion
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Kesehatan
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Makanan
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:30 /user/mahasiswa/etl_output/transaksi_enriched/kategori=Rumah Tangga


**Verifikasi hasil akhir dengan membaca kembali dari HDFS:**

In [17]:
df_final = spark.read.parquet("hdfs://localhost:9000/user/mahasiswa/etl_output/transaksi_enriched")
print("Jumlah baris hasil akhir:", df_final.count())

# Ringkasan cepat sebagai bukti data siap dipakai untuk analisis lebih lanjut
df_final.groupBy("segmen_transaksi").agg(
    count("order_id").alias("jumlah_transaksi"),
    avg("pendapatan").alias("rata_rata_pendapatan")
).orderBy("segmen_transaksi").show()

Jumlah baris hasil akhir: 20000


+----------------+----------------+--------------------+
|segmen_transaksi|jumlah_transaksi|rata_rata_pendapatan|
+----------------+----------------+--------------------+
|          Rendah|            4464|   99639.28001792115|
|          Sedang|            6633|  224445.05759083372|
|          Tinggi|            8903|   398577.4426597776|
+----------------+----------------+--------------------+



**pipeline ETL sudah lengkap** HDFS sebagai penyimpanan sumber & tujuan, Spark sebagai mesin pemrosesan, join & transformasi data, hingga penyimpanan akhir dalam format Parquet yang efisien dan terpartisi.

---

## Menutup SparkSession

In [18]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


---
## Latihan Mandiri

Jalankan ulang cell pembuatan `SparkSession` dari awal modul, lalu baca kembali `data_transaksi_besar.csv` dan `data_pelanggan.json` sebelum mengerjakan latihan berikut.

In [19]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


**Soal 1.** Baca kembali `data_transaksi_besar_parquet` (hasil Sub-bab 6.1.1) menggunakan `spark.read.parquet()`, lalu tampilkan `printSchema()`-nya. Bandingkan dengan skema hasil `inferSchema=True` pada CSV — apakah tipe datanya sama?

In [20]:
# Membaca kembali data Parquet
df_parquet_latihan = spark.read.parquet("data_transaksi_besar_parquet")

print("--- Skema Data Parquet ---")
df_parquet_latihan.printSchema()

print("--- Skema Data CSV (inferSchema=True) ---")
df_transaksi_etl.printSchema()

--- Skema Data Parquet ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

--- Skema Data CSV (inferSchema=True) ---
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



**Soal 2.** Gabungkan `df_transaksi_etl` dengan `df_pelanggan_etl`, lalu tampilkan jumlah transaksi untuk masing-masing nilai `membership` (`Silver`/`Gold`/`Platinum`).

In [21]:
# Menggabungkan df_transaksi_etl dan df_pelanggan_etl berdasarkan customer_id
df_gabungan = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="inner")

# Menghitung jumlah transaksi per kategori membership
df_jumlah_membership = df_gabungan.groupBy("membership").count()

# Menampilkan hasil
df_jumlah_membership.show()

[Stage 5:===========================================================(1 + 0) / 1]

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum| 4016|
|    Silver| 8994|
|      Gold| 6990|
+----------+-----+



**Soal 3.** Tulis ulang hasil gabungan Soal 2 (sebelum di-`groupBy`) ke disk lokal dalam format Parquet, dipartisi berdasarkan `membership`. Tampilkan struktur foldernya dengan `!ls`.

In [22]:
# Menulis hasil gabungan ke penyimpanan lokal dipartisi berdasarkan 'membership'
# Gunakan file:// agar tersimpan pasti di disk/folder lokal, bukan di HDFS
df_gabungan.write.mode("overwrite").partitionBy("membership").parquet("file:///home/zhio/data_transaksi_partisi_membership")

# Menampilkan struktur folder yang dihasilkan di lokal
!ls /home/zhio/data_transaksi_partisi_membership

 _SUCCESS  'membership=Gold'  'membership=Platinum'  'membership=Silver'


**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: mengapa partitioning berdasarkan kolom `kota` cocok untuk skenario kita, namun akan menjadi **pilihan buruk** jika diterapkan pada kolom seperti `order_id` yang nilainya selalu unik di setiap baris? Tulis jawaban pada markdown cell di bawah ini.

*Partisi berdasarkan kolom kota sangat cocok karena memiliki jumlah nilai unik yang sedikit (low cardinality), sehingga hanya menghasilkan beberapa folder terorganisir yang mempercepat kueri (partition pruning). Sebaliknya, mempartisi berdasarkan order_id adalah pilihan buruk karena akan membuat puluhan ribu folder kecil (high cardinality) yang masing-masing hanya berisi satu berkas data (small files problem), yang justru membebani metadata file system dan memperlambat kinerjanya secara drastis.*

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [23]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas6_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dengan skema yang benar | 15% |
| B. Transform - Join | Kedua join (left & inner) diterapkan dengan tepat sesuai kebutuhan masing-masing | 25% |
| C. Transform - Data Quality | Missing value tertangani logis; kolom `ada_ulasan` benar | 20% |
| D. Load | Data tersimpan ke HDFS sebagai Parquet dengan partisi yang benar & terverifikasi | 25% |
| E. Insight Akhir | Analisis tepat & interpretasi bisnis relevan | 15% |


In [27]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round as spark_round

# Inisialisasi SparkSession
spark = SparkSession.builder.appName("Tugas6_ETL").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# 1. Baca data Transaksi (CSV)
df_trx = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
print(f"=== DATA TRANSAKSI: {df_trx.count()} baris ===")
df_trx.printSchema()

# 2. Baca data Produk (JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")
print(f"\n=== DATA PRODUK: {df_produk.count()} baris ===")
df_produk.printSchema()

# 3. Baca data Ulasan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)
print(f"\n=== DATA ULASAN: {df_ulasan.count()} baris ===")
df_ulasan.printSchema()

=== DATA TRANSAKSI: 5000 baris ===
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)


=== DATA PRODUK: 30 baris ===
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)


=== DATA ULASAN: 3500 baris ===
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



In [28]:
# Join Transaksi dengan Ulasan (LEFT JOIN)
df_merged = df_trx.join(df_ulasan, on="order_id", how="left")

# Join dengan Produk (INNER JOIN)
df_merged = df_merged.join(df_produk, on="product_id", how="inner")

# Tambahkan kolom total_pendapatan
df_etl = df_merged.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_etl.show(5)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
o

In [29]:
# 1. Tambah kolom ada_ulasan sebelum na.fill()
df_etl = df_etl.withColumn("ada_ulasan", when(col("rating").isNotNull(), True).otherwise(False))

# 2. Isi nilai NULL pada rating dengan 0
df_etl = df_etl.na.fill({"rating": 0})

df_etl.select("order_id", "nama_produk", "kategori", "rating", "ada_ulasan", "total_pendapatan").show(5)

+--------+-----------+------------+------+----------+----------------+
|order_id|nama_produk|    kategori|rating|ada_ulasan|total_pendapatan|
+--------+-----------+------------+------+----------+----------------+
|     TX0|  Produk-21|  Elektronik|     1|      true|           50000|
|     TX1|   Produk-8|     Fashion|     0|     false|         1500000|
|     TX2|  Produk-25|  Elektronik|     2|      true|          100000|
|     TX3|   Produk-3|     Fashion|     5|      true|          300000|
|     TX4|  Produk-19|Rumah Tangga|     0|     false|          450000|
+--------+-----------+------------+------+----------+----------------+
only showing top 5 rows



Kolom `rating` diisi 0 untuk transaksi yang tidak memiliki ulasan, bukan nilai lain, karena skala rating asli hanya berkisar 1-5 (`np.random.randint(1, 6, ...)` pada pembuatan dataset ulasan). Angka 0 berada di luar rentang skala tersebut, sehingga baris hasil `na.fill()` dapat dibedakan secara jelas dari rating asli — tidak akan pernah tertukar dengan rating 1 (rating terendah yang valid) yang benar-benar diberikan pelanggan. Kolom `ada_ulasan` yang dibuat sebelum `na.fill()` menjadi penanda eksplisit tambahan: setiap analisis rata-rata rating ke depannya wajib memfilter `ada_ulasan == True` terlebih dahulu, sehingga nilai 0 hasil pengisian tidak pernah ikut menurunkan rata-rata rating asli. Mengisi dengan rata-rata rating keseluruhan (seperti pendekatan pada Pertemuan 4) tidak dipilih di sini karena nilai rata-rata tetap berada dalam skala 1-5, sehingga justru mengaburkan makna "belum ada ulasan" tanpa mengecek kolom `ada_ulasan` secara terpisah.

In [30]:
# Tentukan path simpan di HDFS
hdfs_path = "hdfs://localhost:9000/user/zhio/tugas6/hasil_etl"

# Buat direktori di HDFS
!hdfs dfs -mkdir -p /user/zhio/tugas6

# Simpan ke HDFS sebagai Parquet dengan partisi 'kategori'
df_etl.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

print("--- Verifikasi Folder HDFS ---")
!hdfs dfs -ls -R /user/zhio/tugas6/hasil_etl

# Verifikasi membaca kembali dari HDFS
df_verify = spark.read.parquet(hdfs_path)
print("\nJumlah baris terverifikasi di HDFS:", df_verify.count())

--- Verifikasi Folder HDFS ---
-rw-r--r--   1 zhio supergroup          0 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   1 zhio supergroup      16398 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Elektronik/part-00000-0d382768-7fd9-4c6c-b418-fccd63cddc59.c000.snappy.parquet
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   1 zhio supergroup      11162 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Fashion/part-00000-0d382768-7fd9-4c6c-b418-fccd63cddc59.c000.snappy.parquet
drwxr-xr-x   - zhio supergroup          0 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   1 zhio supergroup      10415 2026-09-24 11:48 /user/zhio/tugas6/hasil_etl/kategori=Kesehatan/part-00000-0d382768-7fd9-4c6c-b418-fccd63cddc59.c000.snappy.parquet
drwxr-xr-x   - zhio supergroup     

[Stage 34:===================>                                      (1 + 2) / 3]


Jumlah baris terverifikasi di HDFS: 5000


In [31]:
# Menghitung persentase ulasan per kategori
df_insight = df_etl.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    count(when(col("ada_ulasan") == True, True)).alias("jumlah_diulas")
).withColumn(
    "persentase_diulas", 
    spark_round((col("jumlah_diulas") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_diulas")

df_insight.show()

[Stage 39:>                                                         (0 + 1) / 1]

+------------+---------------+-------------+-----------------+
|    kategori|total_transaksi|jumlah_diulas|persentase_diulas|
+------------+---------------+-------------+-----------------+
|     Makanan|            536|          369|            68.84|
|   Kesehatan|            961|          662|            68.89|
|     Fashion|           1035|          726|            70.14|
|Rumah Tangga|            815|          575|            70.55|
|  Elektronik|           1653|         1168|            70.66|
+------------+---------------+-------------+-----------------+



Kategori **Makanan** memiliki persentase ulasan paling rendah (68,84%) dibanding kategori lain yang berkisar 68,89%-70,66% — selisihnya tipis, tapi Makanan konsisten berada di posisi terbawah. Ini penting bagi tim marketing karena ulasan pelanggan adalah salah satu sinyal kepercayaan (*social proof*) utama yang memengaruhi keputusan pembelian calon pelanggan baru; kategori dengan tingkat ulasan rendah berisiko terlihat kurang meyakinkan di halaman produk dibanding kategori lain, meskipun kualitas produknya sendiri belum tentu bermasalah. Tim marketing bisa menindaklanjuti dengan program insentif ulasan (diskon atau poin loyalitas) khusus pembelian kategori Makanan, atau pengingat otomatis pasca-pembelian yang lebih agresif untuk kategori ini.